# 第71课 · 下一个词怎么选？——贪婪解码与集束搜索（beam search），让 Whisper 开口说话

**学习目标**
- 理解自回归（autoregressive，AR）解码：每步用之前的 token 预测下一个
- 实现贪婪解码（greedy）和简单 beam search（width=2）
- 了解 Whisper 的特殊 token 体系

> **人话**：greedy = 每步最高分；beam = 同时养几条候选再比总分。

← **上一课**　[L70 · Whisper 架构解析](L70_whisper_arch.ipynb)

> 上节课学习了 **Whisper 架构解析**：Log-Mel 输入、Transformer Encoder-Decoder、多任务头。  
> 本课将探讨 **Whisper 解码策略**。

## 本课剧情：Siri 说出第一个字——然后呢？

语音识别的最后一步，是把编码器输出的向量转换成文字。Whisper 不是一次性输出整句话——它是**自回归**的，像人类说话一样，一个词一个词地"想"出来。

**问题**：每一步都有完整的词汇表（Whisper 有 ~51,000 个 BPE token）。怎么从概率分布里选下一个词？

**两种策略**：

| 策略 | 做法 | 类比 |
|---|---|---|
| 贪婪解码 (Greedy) | 每步选概率最大的token | 棋手每步选局部最优走法 |
| Beam Search (宽度W) | 同时追踪W条"候选句子"，每步扩展所有候选 | 棋手同时推演W条路线，保留累计得分最高的 |

**为什么贪婪不够好**？

```
步骤1: P("我")=0.6, P("你")=0.4   → 贪婪选"我"
步骤2: P("我爱")=0.3, P("我恨")=0.7 → 贪婪路径变成"我恨..."
步骤2: P("你好")=0.9               → "你好"路径更好！
```

贪婪在第1步的"局部最优"，可能让第2步走向"全局最差"。Beam Search 同时保留"我"和"你"两条路线，最终选择总对数概率更高的那条。

**本节任务**：用玩具 LM（固定概率分布）分别实现贪婪解码和 Beam Search，数值验证两者的差异。

In [ ]:
import numpy as np

### 预备知识：Logits、Softmax 和数值稳定性

在实现解码算法前，我们需要理解三个核心概念：

**什么是 Logits？**

神经网络的输出层（比如 Transformer 的解码器）不直接输出概率，而是输出 **logits**——原始的、未经处理的分数。你可以把它想象成：

- **概率的"草稿"**：logits = [2.5, 0.1, -1.0, 3.2, ...] 对应词表中不同词的"得分"
- **没有约束的范围**：logits 可以是任意实数（可以很大，可以很小，可以是负数）
- **数学形式**：logits 通常是 embedding 经过线性层的原始输出，记为 $z = W \cdot h + b$

要把 logits 转成[0, 1]的概率，需要 softmax。

**为什么非得用 Softmax 不可？**

Softmax 是一个"把任意数字变成合法概率分布"的魔法函数。它做三件事：

1. **指数运算**（放大差异）：$e^{z_i}$ 让大的数字变得更大，小的数字变得更小
2. **求和归一化**：除以所有指数的和，保证总概率=1
3. **数值稳定化**：减掉最大值后再算，防止大数字造成的溢出

**问题演示：为什么 logits 直接归一化不行？**

```
logits = [1000, 1001, 999]
朴素归一化：概率 = [1000/(1000+1001+999), ...] 

但 e^1000 = 无穷大（计算机算不出来，叫"上溢"overflow）
```

**解决方案：减掉最大值后再算**

```
logits_normalized = logits - max(logits) = [1000, 1001, 999] - 1001 = [-1, 0, -2]

e^(-1) = 0.368，e^0 = 1，e^(-2) = 0.135（都是有限数字）

softmax = [0.368, 1, 0.135] / (0.368+1+0.135) ≈ [0.21, 0.58, 0.08]
```

**关键性质**：这个技巧不改变最终概率，只是数学上等价但数值更安全。

$\text{softmax}(\mathbf{z}) = \text{softmax}(\mathbf{z} - \max(\mathbf{z}))$

（因为分子分母同时乘以 $e^{-\max(\mathbf{z})}$，相消了）


In [ ]:
# 模拟一个玩具 LM：词汇表大小 10，5 步解码
np.random.seed(7)
VOCAB_SIZE = 10
EOT = 9            # end-of-text token id（句子结束标记，类似句子末尾的"。"）
MAX_STEPS = 8

def fake_lm(context: list) -> np.ndarray:
    """玩具语言模型：基于上文输出 logits。（模拟，真实中是 Transformer 前向传播）
    
    参数：
      context: 已生成的 token 序列（列表），例如 [0, 1, 2]
    
    返回：
      logits: shape (VOCAB_SIZE,) 的 1D 数组，每个元素对应一个词的"原始分数"
    
    设计说明：
      - 用 np.random.RandomState(seed) 创建一个确定性的随机生成器
        （seed 相同 → 随机数序列相同，使模型成为"确定的伪随机"）
      - 种子 = sum(context) % 137
        - sum(context) 把所有历史 token 的编号加起来（不同的上文通常得到不同的和）
        - % 137 是为了把和限制在 [0, 137) 范围（137 是一个素数，避免周期过短）
        - 这样设计确保：相同上文 → 相同种子 → 相同 logits（可复现）
      - len(context) >= 4 时增加 EOT logits：模拟真实模型的"长序列倾向早停"行为
        （这是玩具模型的一个简化，帮助演示解码的差异）

      关于 "4 步" 和 "+2.0" 这两个数字——它们是任意的：
        - 这两个参数不是从数学推导出来的，只是作者随手选的两个"够用"的数字，
          目的是让玩具模型在某些轨迹上提前触发 EOT，从而让你能观察到
          贪婪解码 / beam search 处理"提前结束"这件事的逻辑（而不是永远跑满 8 步）。
        - 选 4：因为 MAX_STEPS=8，取一半步数，能保证我们在还没到达步数上限前
          就有机会看到 EOT 被触发；选 3 或 5 同样可以，不影响算法正确性。
        - 选 +2.0：因为 logits 是标准正态分布抽样（大致在 -3~3 之间），
          加 2.0 让 EOT 大概率挤进 softmax 后的高概率区间，但又不是碾压性的
          100%（否则每条 beam 都会在第 4 步整齐结束，就看不出"贪婪 vs beam
          在结束时机上可能不同"这件事了）。换成 +1.0 或 +3.0，也只是让
          EOT 出现得更晚/更早、更不确定/更确定，greedy_decode 和 beam_search
          的代码完全不需要改一行。
        - 换句话说：在真实 Whisper 里，没有"生成到第 4 步就该结束"这种硬编码
          规则——什么时候结束、结束的把握有多大，完全是 Transformer 在海量
          语音-文本对上训练出的权重决定的。这里的两个数字只是我们为了在一个
          没有真实网络的教学环境里，人工制造出"EOT 迟早会出现"的效果，
          方便你专注学习解码算法本身，而不是 LM 的内部结构。
    """
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= 4:
        logits[EOT] += 2.0   # 超过 4 步后更容易结束（提高 EOT 的得分；4 和 2.0 都是任意选的演示参数，见上）
    return logits

def softmax(logits):
    """将 logits（任意实数）转换为概率（[0,1] 且和为 1）。
    
    关键步骤：
      1. x = logits - logits.max()  ← 数值稳定化（防止 exp 溢出）
      2. e = np.exp(x)               ← 指数运算（放大差异）
      3. return e / e.sum()          ← 归一化为概率分布
    
    为什么要减 logits.max()？
      - 如果 logits = [1000, 1001, 999]，np.exp(1000) 会溢出（太大了）
      - 减掉最大值后变成 [-1, 0, -2]，np.exp(-1) ≈ 0.368 是有限的
      - 数学上等价：softmax 的比例结构不变，分子分母同时乘以 e^{-max}，相消了
      - 这个技巧叫"log-sum-exp 稳定化技巧"
    """
    x = logits - logits.max()
    e = np.exp(x)
    return e / e.sum()

### 插曲：EOT 是什么，"第 4 步 +2.0"这两个数字又是哪来的？

**先说 EOT 是什么。** EOT 全称 end-of-text，是词表里一个特殊的 token（这里编号是 9），
作用就像一句话末尾的句号"。"——模型生成到它，就代表"这句话我说完了，可以停了"。
之所以要"特殊对待"它，是因为它不代表任何一个字/词的内容，而是一个"停止信号"：
解码循环每步都要检查"选出来的 token 是不是 EOT"，是的话就跳出循环，不再继续生成。

**再说 `len(context) >= 4` 和 `logits[EOT] += 2.0` 这两个数字。**
读到 `fake_lm` 的代码，很自然会问："为什么是 4 步，不是 3 步或 5 步？为什么是 +2.0，
不是 +1.0 或 +3.0？这两个数字是不是有什么讲究，改了会不会算法就错了？"

**类比一下**：这就像数学应用题里"小明有 5 个苹果"——那个"5"没有任何深意，
换成"3 个苹果"，题目考的知识点（加减法）完全不变。`fake_lm` 里的 "4" 和 "+2.0"
也是同一种性质：它们是**教学场景里随手设定的道具参数**，不是从什么公式推导出来的：

- 选 **4**：因为 `MAX_STEPS = 8`，取一半步数，这样能保证在真正到达步数上限（8 步）
  之前，我们大概率已经能观察到 EOT 被触发一次——这样你才能看到"提前结束"这件事，
  而不是每次都干巴巴地跑满 8 步。选 3、5、甚至 6 都同样可以达到这个教学目的。
- 选 **+2.0**：因为 `rng.randn(...)` 生成的 logits 大致落在 -3 ~ 3 之间，
  加上 2.0 之后，EOT 的 logit 通常会变成众数里最大或接近最大的一个，
  softmax 后它的概率会明显变高，但又不是 100%（如果加 +100，EOT 就会永远
  被选中，贪婪解码和 beam search 就会在完全相同的步数整齐结束，你就看不出
  两种算法在"选择路线"上的差异了——那样这道练习题就失去意义了）。

**关键结论（用代码验证）**：这两个数字只决定"玩具模型多快、多大把握选择结束"，
和 `greedy_decode` / `beam_search` 的算法逻辑本身**没有任何关系**——换成任何
合理的数字，两个解码函数都不需要改一行代码。下面这个 cell 就实际跑一遍，
换几组不同的参数，看 EOT 的概率曲线如何变化。

In [ ]:
### 实验：换掉 "4" 和 "+2.0"，算法会不会跑不动？

# 把 fake_lm 里写死的两个数字变成参数，方便对比不同取值下 EOT 的概率曲线。
def fake_lm_variant(context, eot_threshold=4, eot_bump=2.0):
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= eot_threshold:
        logits[EOT] += eot_bump
    return logits

def eot_prob_curve(eot_threshold, eot_bump, n_steps=8):
    """假装已经生成了 0..n_steps-1 个 token，观察每一步 P(EOT) 的变化。"""
    probs_over_time = []
    for step in range(n_steps):
        context = list(range(step))  # 内容不重要，只用长度模拟"已经生成了几步"
        probs = softmax(fake_lm_variant(context, eot_threshold, eot_bump))
        probs_over_time.append(probs[EOT])
    return probs_over_time

print("课程原版参数：eot_threshold=4, eot_bump=2.0")
for step, p in enumerate(eot_prob_curve(eot_threshold=4, eot_bump=2.0)):
    flag = "  ← 达到阈值，EOT 概率跳高" if step == 4 else ""
    print(f"  已生成 {step} 步 → P(EOT) = {p:.4f}{flag}")

print()
print("换一组参数：eot_threshold=2, eot_bump=1.0（阈值更早、增量更小）")
for step, p in enumerate(eot_prob_curve(eot_threshold=2, eot_bump=1.0)):
    flag = "  ← 达到阈值，EOT 概率跳高" if step == 2 else ""
    print(f"  已生成 {step} 步 → P(EOT) = {p:.4f}{flag}")

print()
print("🔑 关键洞察：")
print("  1. 换了参数，EOT 提前跳高的时机、跳高的幅度都变了，但这只是")
print("     '玩具模型多想结束'这件事本身发生了变化——greedy_decode() 和")
print("     beam_search() 的代码完全没有改，也不需要改。")
print("  2. 这证明了 4 和 +2.0 不是算法的一部分，只是这道练习题的场景设定；")
print("     换成任何合理的正数，解码算法该怎么写还是怎么写。")
print("  3. 真实 Whisper 没有 “第 4 步” 这种硬编码规则——EOT 什么时候该被")
print("     选中，是 Transformer 训练出的权重决定的，这里只是用两个数字")
print("     人工模拟这种'迟早会结束'的效果，方便你专注学解码算法本身。")

## ✏️ 任务 1：贪婪解码

**什么是贪婪解码？**

每一步都选择**当前概率最大的 token**——短视而快速。就像下棋时每步只选局部最强的走法，不往前看多步。

**伪代码直觉**：
```
序列 = [prompt]
循环 max_steps 次：
  1. 用 LM 计算下一步的概率分布
  2. 选概率最大的 token
  3. 加入序列
  4. 如果是 EOT（结束符），停止
```

**为什么叫"贪婪"？**

因为它**只顾当下的收益**，不考虑长远。在某些情况下，一开始概率最高的词可能导致后面句子很差（如本课开头的例子：选"我"的概率 0.6 最高，但后来"我恨"路径比"你好"差）。

**签名**：
```python
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    # 返回: list[int] — 包含 prompt 和生成 token 的完整序列（含 EOT）
```

**3步实现**：

| 步骤 | 操作 | 细节 |
|---|---|---|
| 1 | 初始化序列为 `prompt` 副本 | 不修改输入 |
| 2 | 循环 max_steps 次：调用 `fake_lm(seq)` 取 logits，softmax，`argmax` 得 next_token | 每步 O(V) 时间（V = VOCAB_SIZE） |
| 3 | 若 next_token == eot 则追加 EOT 后 break；否则 append 继续 | 到达 max_steps 自动停止（即使没生成 EOT） |

**验收标准**：
- 输出为 list[int]，长度 ∈ [len(prompt)+1, len(prompt)+max_steps+1]
- 末尾含 EOT token（或达到 max_steps）
- seed=7 的玩具 LM 下结果确定（不随机）

In [ ]:
### 数值演示：为什么要用对数概率？

# 场景：生成一个 100 词的句子，每词的概率都是 0.1（一个困难的任务）
# 问题 1：直接相乘会发生什么？
prob_per_word = 0.1
sequence_length = 100

# 错误做法：直接相乘所有概率
prob_product = prob_per_word ** sequence_length
print(f"❌ 概率相乘：P(句子) = 0.1^{sequence_length} = {prob_product}")
print(f"   结果：{prob_product}（注意：和 0 一样小，Python 显示为 0）\n")

# 正确做法：使用对数概率相加
log_prob_per_word = np.log(prob_per_word)
log_prob_sum = sequence_length * log_prob_per_word
print(f"✅ 对数概率相加：log P(句子) = {sequence_length} × log(0.1)")
print(f"   = {sequence_length} × {log_prob_per_word:.4f}")
print(f"   = {log_prob_sum:.4f}（一个有限的数字，易于比较）\n")

# 对比：两个真实句子的对数概率比较
print("场景：比较两条候选句子哪条更好")
print("-" * 50)
# 句子 A：词概率都是 0.2
log_score_A = 100 * np.log(0.2)
print(f"句子 A：P(词1)=0.2, P(词2)=0.2, ... × 100 词")
print(f"  log 分数 = 100 × log(0.2) = {log_score_A:.4f}\n")

# 句子 B：词概率都是 0.15
log_score_B = 100 * np.log(0.15)
print(f"句子 B：P(词1)=0.15, P(词2)=0.15, ... × 100 词")
print(f"  log 分数 = 100 × log(0.15) = {log_score_B:.4f}\n")

# 比较
print(f"比较：log_score_A ({log_score_A:.4f}) vs log_score_B ({log_score_B:.4f})")
if log_score_A > log_score_B:
    print(f"✅ 结论：句子 A 更好（分数更高 = 概率更高）")
else:
    print(f"✅ 结论：句子 B 更好（分数更高 = 概率更高）")
print()

# 关键洞察
print("🔑 关键洞察：")
print("  1. 对数的一个性质：log(a×b) = log(a) + log(b)")
print("     这样，长序列的概率乘积变成对数的求和（避免下溢）")
print("  2. 对数是严格递增的：如果 a > b，那么 log(a) > log(b)")
print("     所以，比较对数值的大小等价于比较原始概率的大小")
print("  3. 对数值通常是负数（因为概率 < 1）")
print("     但这不影响大小比较（-2.3 < -0.105，意味着 P=0.1 < P=0.9）")


In [ ]:
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    """贪婪解码：每步选概率最大的 token。

    算法步骤提示：
      1. 初始化 tokens = list(prompt)                    # 复制 prompt（不修改输入）
      2. 循环 max_steps 次：
         a. logits = fake_lm(tokens)                    # 神经网络输出原始分数
         b. probs  = softmax(logits)                    # 转换为概率分布 [0,1]，和=1
         c. next_token = int(np.argmax(probs))          # 找最大值所在的索引，转为 int
         d. tokens.append(next_token)                   # 加入序列
         e. 若 next_token == eot：break                 # 遇到句子结束符就停
      3. 返回 tokens
    
    关键点：
      - tokens.append() 修改原列表（有副作用）
      - np.argmax() 返回 numpy.int64，需要 int() 转换后才能存入列表
      - 条件检查（next_token == eot）必须放在 append 之后，否则最后的 EOT 会被遗漏
    """
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

prompt = [0, 1, 2]   # 模拟 [<|startoftranscript|>, <|en|>, <|transcribe|>]
try:
    result = greedy_decode(prompt)
    print(f'贪婪解码结果: {result}')
    # 验证 1：前缀保持不变
    assert result[0:3] == prompt, '前缀应保持不变'
    # 验证 2：以 EOT 结束或达到最大步数（终止条件）
    assert result[-1] == EOT or len(result) - len(prompt) == MAX_STEPS, (
        f'应以 EOT 结束或达到 MAX_STEPS={MAX_STEPS} 步，实际序列: {result}'
    )
    print('✅ 贪婪解码验证通过')
except (NotImplementedError, TypeError):
    result = None
    print('⚠️  greedy_decode 尚未实现，result 设为 None（占位）')

## ✏️ 任务 2：Beam Search（宽度 W=2）

**什么是 Beam Search？**

不是每步都"钉死"一个选择，而是**同时追踪 W 条最有希望的候选句子**（称为"beam"），每步都对它们全部扩展并保留最好的 W 条。类比：国际象棋大师同时推演多条变着，最后选综合得分最高的。

**Beam 的核心数据结构**：每条候选记为 `(log_prob, sequence)`
```python
# 初始状态：1 条 beam，就是 prompt，对数概率为 0（还没生成任何新词）
beams = [(0.0, [0, 1, 2])]
```

**签名**：
```python
def beam_search(prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    # 返回: list[int] — 累计对数概率最高的完整序列
```

**4步算法详解**：

| 步骤 | 操作 | 细节 |
|---|---|---|
| 1 | **扩展候选**：对每条 beam 调 `fake_lm(seq)` → softmax → 对所有 V 个 token 计算 log 概率 | W 条 beam × V 个 token = **W×V 个候选** |
| 2 | **排序和截断**：按累计对数概率降序排序，取前 W 个 | 保留最有希望的 W 条候选 |
| 3 | **处理终止**：若某 beam 末尾 == EOT：移入 completed 列表，不再扩展 | 允许不同 beam 在不同步终止 |
| 4 | **返回最优解**：若 beams 为空或全部完成：从 completed 中取分数最高的 | 返回唯一的最优序列（不是所有 W 个） |

**为什么用累计对数概率而不是单步概率？**

- **单步概率**：只看当前一步的 P(token_i)——可能选到"这一步概率高但后续很差"的词
- **累计概率**：Σ log P(token_1) + log P(token_2) + ... + log P(token_i)——考虑整个路径的质量

累计分数高 = 整个句子每个词的概率都还不错 = 综合质量好。

**完整例子**（W=2, VOCAB_SIZE=3, max_steps=2）：

```
初始：beams = [(0.0, [0, 1])]  

第 1 步扩展：
  beam (0.0, [0,1]) 扩展出 3 个候选
    → (log P(词0), [0,1,0])
    → (log P(词1), [0,1,1])
    → (log P(词2), [0,1,2])
  假设 log 概率分别为 [-1.2, -0.5, -2.0]，取 top-2 保留：
    beams = [(-0.5, [0,1,1]), (-1.2, [0,1,0])]  ← 按分数降序排列

第 2 步扩展：
  beam (-0.5, [0,1,1]) → 新得分 [-1.7, -1.2, -2.5]（加上 -0.5 = 累计）
  beam (-1.2, [0,1,0]) → 新得分 [-2.4, -1.9, -3.2]（加上 -1.2 = 累计）
  全部 2×3=6 个候选，排序并取 top-2：
    beams = [(-1.2, [0,1,1,1]), (-1.7, [0,1,1,0])]
```

**验收标准**：
- seed=7，W=2：beam search 结果与贪婪结果**可能不同**（演示两条路线的差异）
- 返回值是单个序列（对数概率最高的那条），不是所有 W 条

In [ ]:
def beam_search(
    prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT
) -> list:
    """简单 beam search。返回分数最高的序列。

    分数定义：该路径所有 log P(token_i) 之和（越大越好）。
      score(seq) = Σ log P(seq[i] | seq[:i])  for i in range(len(prompt), len(seq))

    为什么 1e-12？
      - softmax 的输出总和 = 1，理论上不应包含 0
      - 但浮点数精度有限，很小的概率可能舍入为 0（例如某个 token 的概率极低）
      - log(0) = -∞（数学上无意义），会导致计算出错
      - 1e-12 是一个很小的"保险系数"，确保 log 的输入总是正数
      - 它不会显著改变高概率 token 的对数值（log(0.3 + 1e-12) ≈ log(0.3)）
      - 但能防止低概率 token 的 log(0) 错误

    算法步骤提示：
      1. 初始化 beams = [(0.0, list(prompt))]   # (累积 log-prob, token 序列)
                   completed = []                # 已结束的候选（末尾是 EOT）
      
      2. 循环 max_steps 次：
         a. candidates = []
         b. 对每个 (score, tokens) in beams：
              logits = fake_lm(tokens)
              probs  = softmax(logits)
              for token_id in range(VOCAB_SIZE):  # ← 这是 V 次（不是 width 次！）
                  new_score = score + np.log(probs[token_id] + 1e-12)  # 累计对数概率
                  candidates.append((new_score, tokens + [token_id]))
         c. 按 score 降序排列 candidates（最高分在前）
            候选总数 = len(beams) × VOCAB_SIZE（例 width=2, V=10 → 20 个候选）
         d. 取前 width 个候选作为新的 beams（例 width=2 → 保留前 2 个）
         e. 对新的 beams 检查末尾是否为 EOT：
              - 若是 EOT：移入 completed（这条 beam 已完成，不再扩展）
              - 若不是：保留在 beams 中（继续扩展）
         f. 若 beams 为空（所有候选都以 EOT 结束）：break

      3. 若循环提前 break（所有 beam 完成）或达到 max_steps：
         从 completed + beams 中找分数最高的序列返回
    """
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

try:
    beam_result = beam_search(prompt, width=2)
    print(f'Beam search 结果 (W=2): {beam_result}')
    if result is not None:
        print(f'贪婪结果:               {result}')
    print('（两者可能不同——beam search 探索更多路径）')

    # 验证 1：返回值以 prompt 为前缀
    assert beam_result[:len(prompt)] == prompt, 'beam_result 应以 prompt 为前缀'

    # 验证 2：beam search log-prob ≥ greedy log-prob（最优性保证）
    if result is not None:
        def compute_logprob(seq):
            score = 0.0
            tokens = list(prompt)
            for tok in seq[len(prompt):]:
                logits = fake_lm(tokens)
                probs = softmax(logits)
                score += np.log(probs[tok] + 1e-12)
                tokens.append(tok)
            return score
        beam_score   = compute_logprob(beam_result)
        greedy_score = compute_logprob(result)
        assert beam_score >= greedy_score - 1e-6, (
            f'beam search 分数 ({beam_score:.4f}) 应 ≥ greedy 分数 ({greedy_score:.4f})'
        )
    print('✅ Beam search 验证通过')
except (NotImplementedError, TypeError):
    print('⚠️  beam_search 尚未实现，跳过验证')

## 🔗 Aurora 连接

本课实现的两个解码算法与 Aurora 代码库的对应关系：

| 本课实现 | Aurora 模块 | 说明 |
|---|---|---|
| `greedy_decode` | `aurora.llm.greedy_decode`（`src/aurora/llm/sample.py`） | src 版是单步 argmax（对一帧 logits 选最大）；本课把它扩展成完整的自回归解码循环 |
| `beam_search` | 本课 notebook 实现 | src 中尚未收录 beam search；Whisper 论文长音频转写设定用 beam_size=5 |

`aurora.llm` 目前提供单步的 `greedy_decode` 与 `top_k_sample` / `top_p_sample`；
本课的贪婪循环与 beam search 演示如何把"单步选 token"组装成完整序列解码。
L86 将在此基础上系统学习 temperature / top-k / top-p 采样（`aurora.llm.top_k_sample`）。


In [ ]:
### Python 细节速成班：argmax、list 拼接、对数比较

# 问题 1：np.argmax() 返回什么？
probs = np.array([0.1, 0.6, 0.2, 0.1])
idx = np.argmax(probs)
print(f"概率数组：{probs}")
print(f"np.argmax(probs) = {idx}")
print(f"类型：{type(idx)}（numpy int64）")
print(f"int(idx) = {int(idx)}（转换为 Python int）")
print(f"argmax 找的是最大值所在的**索引**（第几个元素），不是值本身")
print()

# 问题 2：list 拼接会修改原列表吗？
tokens = [0, 1, 2]
new_token = 3
new_sequence = tokens + [new_token]  # ← 这创建一个新列表
print(f"原列表 tokens = {tokens}")
print(f"新列表 tokens + [3] = {new_sequence}")
print(f"原列表是否被修改？{tokens}（没有！）")
print(f"append() 会修改原列表，但 + 创建新列表")
print()

# 问题 3：对数概率的大小比较
log_prob_A = -45.3
log_prob_B = -42.1
print(f"两条路线的对数概率：")
print(f"  路线 A：log P(seq_A) = {log_prob_A}")
print(f"  路线 B：log P(seq_B) = {log_prob_B}")
print(f"哪条更好？")
if log_prob_B > log_prob_A:
    print(f"  ✅ 路线 B 更好（{log_prob_B} > {log_prob_A}）")
    print(f"  原因：-42.1 更接近 0，意味着概率更高")
else:
    print(f"  ✅ 路线 A 更好（{log_prob_A} > {log_prob_B}）")
print(f"记住：对于负数，-2 > -5（更接近 0 = 值更大）")
print()

# 问题 4：1e-6 容差是什么意思？
score_a = -42.1234567
score_b = -42.1234560
tolerance = 1e-6
print(f"验证：beam_score >= greedy_score - 1e-6")
print(f"  beam_score = {score_a:.10f}")
print(f"  greedy_score = {score_b:.10f}")
print(f"  greedy_score - 1e-6 = {score_b - tolerance:.10f}")
print(f"  检查：{score_a} >= {score_b - tolerance} ？")
if score_a >= score_b - tolerance:
    print(f"  ✅ 通过（允许微小的浮点误差）")
else:
    print(f"  ❌ 不通过")
print(f"为什么要 - 1e-6？")
print(f"  因为浮点运算有舍入误差，两个理论上相等的数值可能相差 1e-15 ~ 1e-10")
print(f"  给个小的容差（1e-6）能容许这些舍入误差，避免虚假失败")


### Beam Search 的最优性保证

**问题**：Beam Search 真的能找到最优解吗？

**答案**（分两个层面）：

1. **相对于贪婪解码**：✅ 是的，Beam Search 分数 ≥ 贪婪分数
   - 原因：贪婪解码是 Beam Search 的特例（W=1）
   - Beam Search(W=2) 同时考虑贪婪的选择 + 其他选择
   - 所以 Beam(W) 的最优路线分数 ≥ Beam(W=1) = 贪婪的路线分数

2. **相对于全局最优**：❌ 不一定，但通常很接近
   - 全局最优 = 遍历 V^T 条所有可能序列，选分数最高的（指数级别，不可行）
   - Beam Search(W) 只遍历 W×V^T 条序列（W 是宽度，通常很小）
   - 所以 Beam 不保证全局最优，但在 W 足够大时通常很接近
   
   **直观比喻**：
   - 全局最优 = 国际象棋穷举所有可能走法（$30^{100}$ 种，不可能）
   - Beam W=5 = 同时推演 5 条最有希望的变着（实际可行，通常效果很好）
   - W 越大，找到最优解的概率越高，但计算代价也越高

**代码体现**：
```python
# 验证代码检查的是"相对最优"（贪婪的下界）
assert beam_score >= greedy_score - 1e-6
# 意思是：Beam 分数 ≥ 贪婪分数（在浮点误差范围内）
```

这个检查**不能证明 Beam 找到全局最优**，只能证明它至少和贪婪一样好（通常更好）。


## 本课收束

| 策略 | 复杂度 | 特点 |
|---|---|---|
| 贪婪 | O(T×V) | 快，局部最优 |
| Beam W=2 | O(T×W×V) | 更好，代价是 W 倍计算 |
| Top-k / Top-p | O(T×V) | 引入随机性，避免重复（见 L86）|

Whisper 温度 0 时默认贪婪解码；论文的长音频转写设定用 beam_size=5 + length_penalty 换取更高质量。

下一步 L72：在真实数据集上用 LoRA 微调 Whisper。

## ✏️ 白板挑战：解码策略手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：贪婪解码的时间复杂度是多少？设序列长度 T，词表大小 V。  
（每步 argmax 是 O(V)，共 T 步，总计 O(T×V)）

**问 2**：Beam Search（宽度 W）每步扩展后有多少候选序列？取 top-W 后剩几条？  
（扩展后 W×V 条候选；取 top-W 后保留 W 条）

**问 3**：若 W=1，Beam Search 退化成什么？  
（退化为贪婪解码——每步只保留1条beam，等价于argmax）

**问 4**：为什么 Beam Search 用对数概率相加，而不是概率相乘？  
（避免浮点下溢：0.1^100 ≈ 0，但 100×log(0.1) = -230，数值安全；加法也更快）

**问 5**：Whisper 推理时默认用哪种解码策略？为什么？  
（openai-whisper 在温度 0 时默认贪婪解码；论文的长音频转写设定用 beam_size=5。贪婪更快，beam search 更准，可按需在速度和质量间权衡）

推导完成后运行下方格验证。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：贪婪复杂度演示（O(T×V)步骤计数）
try:
    import time
    np.random.seed(7)
    prompt_test = [0]
    t0 = time.perf_counter()
    result_greedy = greedy_decode(prompt_test, max_steps=MAX_STEPS)
    t1 = time.perf_counter()
    assert isinstance(result_greedy, list), "greedy_decode应返回list"
    assert result_greedy[0] == 0, "序列应以prompt开头"
    # 终止条件：以 EOT 结束，或到达 MAX_STEPS（本玩具 LM 下 [0] 起步不一定命中 EOT）
    assert result_greedy[-1] == EOT or len(result_greedy) - len(prompt_test) == MAX_STEPS, (
        f"序列应以 EOT 结束或达到 MAX_STEPS={MAX_STEPS} 步，实际: {result_greedy}"
    )
    print(f"Q1 ✅  greedy_decode: {len(result_greedy)}个token，每步O(V={VOCAB_SIZE})，总O({len(result_greedy)*VOCAB_SIZE})")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  贪婪复杂度=O(T×V)，T=序列长度，V={VOCAB_SIZE}（待实现后验证）")

# 问2：beam search候选数
try:
    np.random.seed(7)
    result_beam = beam_search([0], width=2, max_steps=MAX_STEPS)
    assert isinstance(result_beam, list), "beam_search应返回list"
    assert result_beam[0] == 0, "序列应以prompt开头"
    W = 2
    print(f"Q2 ✅  Beam W={W}: 每步扩展后{W}×{VOCAB_SIZE}={W*VOCAB_SIZE}候选，取top-{W}保留{W}条")
    # 检查是否与贪婪不同（演示beam的价值）
    if result_greedy != result_beam:
        print(f"       ✨ beam结果{result_beam} ≠ 贪婪结果{result_greedy}（beam找到了更好路径！）")
    else:
        print(f"       beam结果与贪婪相同（玩具LM下有时相同，真实LM通常不同）")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  Beam W=2: 每步W×V=2×{VOCAB_SIZE}={2*VOCAB_SIZE}候选，取top-2（待实现后验证）")

# 问3：W=1退化
try:
    np.random.seed(7)
    result_w1 = beam_search([0], width=1, max_steps=MAX_STEPS)
    # W=1时beam应与greedy结果相同
    assert result_w1 == result_greedy, f"W=1时beam={result_w1}应等于greedy={result_greedy}"
    print(f"Q3 ✅  W=1 beam search = greedy decode: {result_w1}")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  W=1退化为贪婪（只保留1条beam = argmax）（待实现后验证）")
except AssertionError as e:
    print(f"Q3 ⚠️  {e}（检查beam_search(width=1)是否正确实现）")

# 问4：对数概率（数值验证）
probs_q4 = np.array([0.1] * 10)   # 简单均匀分布
log_probs = np.log(probs_q4 + 1e-10)
assert np.all(np.isfinite(log_probs)), "log_probs应全有限"
# 100步概率乘积 vs 对数概率相加
prob_product = np.prod(probs_q4[:5])
log_sum = np.sum(log_probs[:5])
print(f"Q4 ✅  概率乘积={prob_product:.2e}（接近下溢），对数和={log_sum:.3f}（数值安全）")

# 问5：Whisper默认策略（概念验证）
print(f"Q5 ✅  Whisper温度0时默认贪婪解码；论文长音频设定用beam_size=5（可用--beam_size调节）；贪婪=最快，beam=最准")
print("\n🎉 Whisper解码策略白板挑战通过！")

In [ ]:
# ✏️ 本课自评
l71_review = {
    "greedy_vs_beam_tradeoff":  None,  # 理解贪婪局部最优vs beam全局搜索的权衡？True/False
    "log_prob_rationale":       None,  # 理解为什么用对数概率相加（避免下溢）？True/False
    "greedy_decode_impl":       None,  # greedy_decode()实现正确，含EOT，seed=7结果确定？True/False
    "beam_search_impl":         None,  # beam_search(width=2)实现正确，返回最高累计对数概率序列？True/False
    "whiteboard_passed":        None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l71_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l71_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L71 全部通关！进入 L72：Whisper 微调')

---

→ **下一课**　[L72 · Whisper 微调](L72_whisper_finetune.ipynb)

> 下节课将学习 **Whisper 微调**：LoRA 低秩注入 vs 全参数，中文 / 方言数据适配实战。